# ArcFace Training for Migros Dataset v1

This notebook trains an ArcFace model on the `migros_dataset_v1`.
It implements the necessary CSV parsing to map Product IDs to Names and load annotations for testing.

**Architecture:** ResNet34 + ArcFace Head + HAL (Hierarchical Auxiliary Loss).

In [ ]:
import os
import cv2
import csv
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.notebook import tqdm
import math
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
class Config:
    # Paths
    ROOT_DIR = r"c:\Users\sinan\Desktop\grocery_detection\grocery_project\datasets\migros_dataset_v1"
    TRAINING_DIR = os.path.join(ROOT_DIR, "Training")
    TESTING_DIR = os.path.join(ROOT_DIR, "Testing")
    ANNOTATIONS_PATH = os.path.join(ROOT_DIR, "Annotations", "_annotations.csv")
    ID_MAPPING_PATH = os.path.join(ROOT_DIR, "Annotations", "SDP_Product&ID_Dataset_fix.csv")
    CHECKPOINT_DIR = r"c:\Users\sinan\Desktop\grocery_detection\grocery_project\classification\checkpoints_migros_v1"
    
    # Model
    FEATURE_DIM = 512
    ARCFACE_SCALE = 64
    ARCFACE_MARGIN = 0.5
    HAL_WEIGHT = 2.0  # Hierarchical Loss Weight
    
    # Training
    NUM_EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.01
    MIN_LR = 0.0001
    WEIGHT_DECAY = 1e-4
    
    # Data
    IMAGE_SIZE = 224
    NUM_WORKERS = 0
    SAMPLES_PER_EPOCH = 50000  # Number of samples to synthesize/iterate per epoch

config = Config()
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

## 1. Data Parsing & Mapping
Load the ID-to-Name mapping and find training images.

In [ ]:
def load_id_mapping(csv_path):
    """
    Load mapping from ID to Product Name.
    Assumes CSV format: ID,Name
    """
    id_to_name = {}
    name_to_id = {}
    
    # Read CSV, assume no header or check first row
    # Based on inspection: '1,Lipton...' 
    try:
        df = pd.read_csv(csv_path, header=None)
        # If first column is not int, maybe it has header
        if not str(df.iloc[0,0]).isdigit():
             df = pd.read_csv(csv_path)
             
        for _, row in df.iterrows():
            try:
                pid = int(row[0])
                name = str(row[1]).strip()
                id_to_name[pid] = name
                name_to_id[name] = pid
            except ValueError:
                continue
    except Exception as e:
        print(f"Error reading ID mapping: {e}")
            
    return id_to_name, name_to_id

print("Loading ID mapping...")
id_to_name, name_to_id = load_id_mapping(config.ID_MAPPING_PATH)
print(f"Loaded {len(id_to_name)} product mappings.")

In [ ]:
def find_training_images(root_dir, name_to_id):
    """
    Walk through Training directory.
    Match filename (without extension) to Product Name.
    Returns list of dicts with path, real_id, category, name.
    """
    training_samples = []
    matched_count = 0
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                name_stem = os.path.splitext(file)[0]
                
                real_id = name_to_id.get(name_stem)
                
                if real_id is None:
                    # Optional: Print first few mismatches to debug
                    # print(f"Warning: No ID found for {name_stem}")
                    continue
                
                matched_count += 1
                
                # Category is the immediate parent folder name
                category = os.path.basename(root)
                
                training_samples.append({
                    'path': os.path.join(root, file),
                    'real_id': real_id,
                    'category': category,
                    'name': name_stem
                })
                
    return training_samples

print("Scanning training images...")
train_samples_raw = find_training_images(config.TRAINING_DIR, name_to_id)
print(f"Found {len(train_samples_raw)} training images.")

In [ ]:
def parse_annotations(ann_file, test_root):
    """
    Parse _annotations.csv for testing.
    Expected Columns: filename,width,height,class,xmin,ymin,xmax,ymax
    """
    test_samples = []
    
    try:
        # Use on_bad_lines='skip' if needed, generally pandas handles consistent CSVs well
        df = pd.read_csv(ann_file)
        
        # Normalize column names just in case
        df.columns = [c.strip() for c in df.columns]
        
        for _, row in df.iterrows():
            filename = row['filename']
            # Assuming 'class' column holds the ID
            class_id = int(row['class'])
            
            # BBox
            bbox = [
                int(row['xmin']),
                int(row['ymin']),
                int(row['xmax']),
                int(row['ymax'])
            ]
            
            # Find image path (check recursive if flat path fails)
            img_path = os.path.join(test_root, filename)
            
            # If not found directly, assume it might be valid if we trust CSV
            # Or basic existence check can be done in dataset
            
            test_samples.append({
                'path': img_path,
                'bbox': bbox,
                'real_id': class_id
            })
            
    except Exception as e:
        print(f"Error parsing annotations: {e}")
        
    return test_samples

print("Parsing test annotations...")
test_samples_raw = parse_annotations(config.ANNOTATIONS_PATH, config.TESTING_DIR)
print(f"Found {len(test_samples_raw)} test samples.")

In [ ]:
## Build Internal Indexes
# ArcFace requires contiguous class indices 0..N-1

unique_ids = set()
for s in train_samples_raw:
    unique_ids.add(s['real_id'])
    
# Add test IDs to ensure we cover everything, though generally training set defines the model
for s in test_samples_raw:
    unique_ids.add(s['real_id'])
    
sorted_ids = sorted(list(unique_ids))
real_to_model = {rid: i for i, rid in enumerate(sorted_ids)}
model_to_real = {i: rid for i, rid in enumerate(sorted_ids)}

NUM_CLASSES = len(sorted_ids)

# Categories for HAL
# Map model_class_idx -> category_idx
categories = set(s['category'] for s in train_samples_raw)
cat_to_idx = {c: i for i, c in enumerate(sorted(list(categories)))}
NUM_CATEGORIES = len(cat_to_idx)

# Build class_to_category mapping (model_idx -> category_name)
# Only available for classes present in training set
class_to_category = {}
for s in train_samples_raw:
    mid = real_to_model[s['real_id']]
    class_to_category[mid] = s['category']

print(f"Total Classes: {NUM_CLASSES}")
print(f"Total Categories: {NUM_CATEGORIES}")

In [ ]:
class GroceryTrainDataset(Dataset):
    """
    Training Dataset.
    Because we might have 1 shot (Reference Only), we cache images and apply strong augmentation.
    """
    def __init__(self, samples, real_to_model, cat_to_idx, transform=None, epoch_size=50000):
        self.samples = samples # List of dicts
        self.real_to_model = real_to_model
        self.cat_to_idx = cat_to_idx
        self.transform = transform
        self.epoch_size = epoch_size
        
        # Cache images in memory (assuming dataset fits in RAM)
        self.cache = {}
        print(f"Caching {len(samples)} training images...")
        for s in tqdm(samples):
            try:
                img = Image.open(s['path']).convert('RGB')
                self.cache[s['path']] = img
            except Exception as e:
                print(f"Failed to load: {s['path']}")
        
    def __len__(self):
        return self.epoch_size
    
    def __getitem__(self, idx):
        # Randomly sample a product
        s_idx = np.random.randint(0, len(self.samples))
        sample = self.samples[s_idx]
        
        path = sample['path']
        img = self.cache.get(path)
        
        if img is None:
             # Fallback
             img = Image.new('RGB', (224, 224), (128,128,128))
        else:
             img = img.copy()
             
        if self.transform:
            img = self.transform(img)
            
        # Targets
        class_idx = self.real_to_model[sample['real_id']]
        cat_name = sample['category']
        cat_idx = self.cat_to_idx.get(cat_name, 0)
        
        return img, class_idx, cat_idx

class GroceryTestDataset(Dataset):
    def __init__(self, samples, real_to_model, transform=None):
        self.samples = samples
        self.real_to_model = real_to_model
        self.transform = transform
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        try:
            image = Image.open(sample['path']).convert('RGB')
            x1, y1, x2, y2 = sample['bbox']
            
            # Crop
            w, h = image.size
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            
            if x2 > x1 and y2 > y1:
                image = image.crop((x1, y1, x2, y2))
        except:
            image = Image.new('RGB', (224, 224), (128,128,128))
            
        if self.transform:
            image = self.transform(image)
            
        class_idx = self.real_to_model.get(sample['real_id'], -1)
        
        return image, class_idx

In [ ]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(20),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Datasets
train_dataset = GroceryTrainDataset(
    train_samples_raw,
    real_to_model,
    cat_to_idx,
    transform=train_transform,
    epoch_size=config.SAMPLES_PER_EPOCH
)

test_dataset = GroceryTestDataset(
    test_samples_raw,
    real_to_model,
    transform=test_transform
)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=config.NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=config.NUM_WORKERS)

In [ ]:
## 2. Model (ArcFace + HAL)
Standard ArcFace implementation with ResNet34 backbone.

In [ ]:
class ArcFaceHead(nn.Module):
    def __init__(self, in_features, num_classes, scale=64.0, margin=0.5):
        super().__init__()
        self.scale = scale
        self.margin = margin
        self.num_classes = num_classes
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.current_margin = 0.0

    def set_margin(self, margin):
        self.current_margin = min(margin, self.margin)

    def forward(self, embeddings, labels):
        normalized_weights = F.normalize(self.weight, dim=1)
        cosine = F.linear(embeddings, normalized_weights)
        
        m = self.current_margin
        if m > 0:
            cos_m, sin_m = math.cos(m), math.sin(m)
            th = math.cos(math.pi - m)
            mm = math.sin(math.pi - m) * m
            
            sine = torch.sqrt(1.0 - cosine.pow(2).clamp(0, 1))
            phi = cosine * cos_m - sine * sin_m
            phi = torch.where(cosine > th, phi, cosine - mm)
            
            one_hot = F.one_hot(labels, self.num_classes).float()
            output = one_hot * phi + (1 - one_hot) * cosine
        else:
            output = cosine
            
        return output * self.scale
        
    def get_proxies(self):
        return F.normalize(self.weight, dim=1)

class HALHead(nn.Module):
    def __init__(self, class_to_category, num_categories, scale=64.0):
        super().__init__()
        self.scale = scale
        self.num_categories = num_categories
        
        # Map cat_idx -> list of class_idx
        self.category_to_classes = defaultdict(list)
        for cls_idx, cat_name in class_to_category.items():
            # We need cat_idx
            if cat_name in cat_to_idx:
                c_idx = cat_to_idx[cat_name]
                self.category_to_classes[c_idx].append(cls_idx)

    def forward(self, embeddings, cat_labels, class_proxies):
        cat_proxies_list = []
        for cat_idx in range(self.num_categories):
             class_indices = self.category_to_classes[cat_idx]
             if class_indices:
                 cat_proxy = class_proxies[class_indices].mean(dim=0)
                 cat_proxy = F.normalize(cat_proxy, dim=0)
             else:
                 cat_proxy = torch.zeros(class_proxies.size(1), device=class_proxies.device)
             cat_proxies_list.append(cat_proxy)
             
        cat_proxies_tensor = torch.stack(cat_proxies_list)
        logits = self.scale * F.linear(embeddings, cat_proxies_tensor)
        loss = F.cross_entropy(logits, cat_labels)
        return loss

class ProductRecognitionModel(nn.Module):
    def __init__(self, num_classes, num_categories, class_to_category, embedding_dim=512):
        super().__init__()
        resnet = models.resnet34(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        self.embedding = nn.Sequential(
            nn.Linear(512, embedding_dim),
            nn.BatchNorm1d(embedding_dim)
        )
        
        self.arcface = ArcFaceHead(embedding_dim, num_classes)
        self.hal = HALHead(class_to_category, num_categories)
        
    def get_embeddings(self, x):
        features = self.backbone(x).flatten(1)
        embeddings = self.embedding(features)
        return F.normalize(embeddings, dim=1)
        
    def forward(self, x, class_labels=None, cat_labels=None):
        embeddings = self.get_embeddings(x)
        
        if class_labels is not None:
            logits = self.arcface(embeddings, class_labels)
            
            hal_loss = 0
            if cat_labels is not None:
                 proxies = self.arcface.get_proxies()
                 hal_loss = self.hal(embeddings, cat_labels, proxies)
                 
            return embeddings, logits, hal_loss
        return embeddings

model = ProductRecognitionModel(NUM_CLASSES, NUM_CATEGORIES, class_to_category, embedding_dim=config.FEATURE_DIM).to(device)
optimizer = optim.SGD(model.parameters(), lr=config.LEARNING_RATE, momentum=0.9, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS, eta_min=config.MIN_LR)
scaler = torch.cuda.amp.GradScaler()

In [ ]:
## 3. Training Loop

In [ ]:
def train_epoch(epoch):
    model.train()
    
    # Margin scheduling
    warmup = 5
    if epoch < warmup:
         m = config.ARCFACE_MARGIN * epoch / warmup
    else:
         m = config.ARCFACE_MARGIN
    model.arcface.set_margin(m)
    
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for images, class_idxs, cat_idxs in pbar:
        images = images.to(device)
        class_idxs = class_idxs.to(device)
        cat_idxs = cat_idxs.to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            embeddings, logits, hal_loss = model(images, class_idxs, cat_idxs)
            ce_loss = F.cross_entropy(logits, class_idxs)
            loss = ce_loss + config.HAL_WEIGHT * hal_loss
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        # Acc
        preds = logits.argmax(dim=1)
        correct += (preds == class_idxs).sum().item()
        total += images.size(0)
        
        pbar.set_postfix({'Loss': loss.item(), 'Acc': correct/total, 'Margin': m})
        
    return total_loss / len(train_loader), correct / total

for epoch in range(config.NUM_EPOCHS):
    loss, acc = train_epoch(epoch)
    scheduler.step()
    print(f"Epoch {epoch+1} - Loss: {loss:.4f}, Acc: {acc:.4f}")
    
    # Save
    if (epoch + 1) % 5 == 0:
        torch.save(model.state_dict(), os.path.join(config.CHECKPOINT_DIR, f"arcface_epoch_{epoch+1}.pth"))